# 13_lightweight_synthesis_for_mentor_report_260515

Mentor reporting synthesis only. No new modeling, SHAP, Optuna, thresholds, or segmentation.

In [1]:
from pathlib import Path
from datetime import datetime
import subprocess
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display, Markdown

STEP = '13_lightweight_synthesis_for_mentor_report_260515'
EXPECTED_ROOTS = {Path('C:/Code/ott-churn-prediction').resolve()}
ROOT = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()).resolve()
PARK = (ROOT / 'park.ingyeom').resolve()
NOTEBOOK_PATH = (PARK / 'notebook' / STEP / f'{STEP}.ipynb').resolve()

if ROOT not in EXPECTED_ROOTS:
    raise SystemExit(f'Repo root mismatch: {ROOT}')
if not PARK.exists():
    raise SystemExit(f'park.ingyeom folder missing: {PARK}')

def inside_park(path):
    path = Path(path).resolve()
    try:
        path.relative_to(PARK)
        return True
    except ValueError:
        return False

def require_inside_park(path):
    path = Path(path).resolve()
    if not inside_park(path):
        raise SystemExit(f'Path outside park.ingyeom blocked: {path}')
    return path

def rel(path):
    return str(Path(path).resolve().relative_to(PARK)).replace('\\', '/')

def read_csv(path):
    path = require_inside_park(path)
    if '_data' in path.parts:
        raise SystemExit(f'_data read blocked: {path}')
    return pd.read_csv(path)

def write_csv(df, path):
    path = require_inside_park(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

def write_text(text, path):
    path = require_inside_park(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
    return path

def all_status_pass(path):
    path = Path(path)
    if not path.exists():
        return False
    df = pd.read_csv(path)
    status_cols = [c for c in df.columns if c.lower() == 'status']
    if status_cols:
        s = df[status_cols[0]].astype(str).str.upper().str.strip()
        return len(s) > 0 and bool((s == 'PASS').all())
    passed_cols = [c for c in df.columns if c.lower() == 'passed']
    if passed_cols:
        s = df[passed_cols[0]].astype(str).str.upper().str.strip()
        return len(s) > 0 and bool(s.isin(['TRUE', 'PASS', '1', 'YES']).all())
    return False

def choose_output_folder(base):
    base = require_inside_park(base)
    if base.exists() and any(x.is_file() for x in base.iterdir()):
        folder = base / ('run_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    else:
        folder = base
    folder.mkdir(parents=True, exist_ok=True)
    return folder.resolve()

def detect_latest_valid_folder(base, required_files, final_check_name):
    base = require_inside_park(base)
    candidates = []
    if base.exists():
        candidates.append(base)
        candidates.extend([p for p in base.iterdir() if p.is_dir()])
    candidates = sorted(candidates, key=lambda p: (p.stat().st_mtime, p.name), reverse=True)
    for folder in candidates:
        if all((folder / f).exists() for f in required_files) and all_status_pass(folder / final_check_name):
            return folder.resolve()
    return None

OUTPUT_FOLDER = choose_output_folder(PARK / 'reports' / 'brief' / STEP)
FIGURE_FOLDER = choose_output_folder(PARK / 'reports' / 'figures' / STEP)
ZIP_PATH = require_inside_park(PARK / 'zip' / f'{STEP}_review_package.zip')
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)

paths = {
    'note_md': PARK / 'note.md',
    '05b_dir': PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513',
    '06': PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513',
    '08b': PARK / 'reports' / 'eda' / '08b_promotion_vs_nonpromotion_eda_audit_patch_260513',
    '09': PARK / 'reports' / 'eda' / '09_promotion_repurchase_2x2_eda_260513',
    '09b': PARK / 'reports' / 'audits' / '09b_raw_view_window_validation_260514' / 'run_20260514_130402',
    '10_base': PARK / 'reports' / 'eda' / '10_feature_eda_260513',
    '11b_base': PARK / 'reports' / 'models' / '11b_baseline_growth_history_ladder_fix_260514',
    '11b_semantic': PARK / 'reports' / 'audits' / '11b_semantic_validation_and_interpretation_patch_260514',
    '12c_base': PARK / 'reports' / 'models' / '12_model_baseline_comparison_canonical_260514',
}

req_06 = ['06_cohort_summary.csv', '06_primary_main_cohort_index.csv', '06_primary_main_cohort_conservative_features.csv', '06_final_checks.csv']
req_08b = ['08b_decision_summary.csv', '08b_interpretation_guardrail.csv', '08b_promotion_feature_difference_negative_finding.csv']
req_09 = ['09_2x2_cohort_definition.csv', '09_within_promotion_target_difference_summary.csv', '09_within_nonpromotion_target_difference_summary.csv', '09_08_vs_09_contrast_summary.csv', '09_final_checks.csv']
req_09b = ['09b_final_checks.csv', '09b_raw_view_relative_day_distribution.csv', '09b_core_usage_recalculation_comparison.csv', '09b_day21_plus_leakage_contrast_test.csv', '09b_window_validation_decision.csv']
req_10 = ['10_final_checks.csv', '10_eda_insight_candidates.csv', '10_focus_feature_deep_dive_summary.csv', '10_retention_third_week_audit.csv', '10_week_progression_pattern_audit.csv', '10_handoff_to_11_and_17.csv']
req_11b = ['11b_final_checks.csv', '11b_best_baseline_by_scope.csv', '11b_ladder_growth_summary.csv', '11b_cv_summary_metrics.csv', '11b_open_risks_for_next_steps.csv']
req_11b_semantic = ['11b_semantic_final_checks.csv', '11b_canonical_status_decision.csv', '11b_ladder_interpretation_guardrail.csv', '11b_handoff_to_12_semantic_requirements.csv']
req_12c = ['12c_final_checks.csv', '12c_model_comparison_summary.csv', '12c_operating_metrics_at_k.csv', '12c_candidate_selection_by_scope.csv', '12c_stability_aware_candidate_by_scope.csv', '12c_vs_11b_baseline_comparison.csv', '12c_calibration_decile_summary.csv', '12c_safe_unsafe_wording.csv', '12c_open_risks_for_next_steps.csv']

folder_10 = detect_latest_valid_folder(paths['10_base'], req_10, '10_final_checks.csv')
folder_11b = detect_latest_valid_folder(paths['11b_base'], req_11b, '11b_final_checks.csv')
folder_12c = detect_latest_valid_folder(paths['12c_base'], req_12c, '12c_final_checks.csv')
folder_12c_status = 'present_and_pass' if folder_12c else 'missing_or_not_all_pass'

required_groups = []
for label, folder, files in [
    ('06', paths['06'], req_06), ('08b', paths['08b'], req_08b), ('09', paths['09'], req_09),
    ('09b', paths['09b'], req_09b), ('10', folder_10, req_10), ('11b', folder_11b, req_11b),
    ('11b_semantic', paths['11b_semantic'], req_11b_semantic)]:
    if folder is None:
        required_groups.append({'source': label, 'path': '', 'exists': False})
    else:
        for f in files:
            required_groups.append({'source': label, 'path': rel(Path(folder) / f), 'exists': (Path(folder) / f).exists()})
if folder_12c:
    for f in req_12c:
        required_groups.append({'source': '12c', 'path': rel(folder_12c / f), 'exists': (folder_12c / f).exists()})

must_exist_ok = all(x['exists'] for x in required_groups if x['source'] != '12c')
if not must_exist_ok:
    missing = [x for x in required_groups if not x['exists'] and x['source'] != '12c']
    raise SystemExit(f'Missing required non-12c inputs: {missing}')

archive_dir = PARK / '_archive'
archive_manifests = []
if archive_dir.exists():
    archive_manifests = sorted([p.resolve() for p in archive_dir.rglob('*manifest*.csv')])

checks = {
    '06_final_checks_pass': all_status_pass(paths['06'] / '06_final_checks.csv'),
    '09_final_checks_pass': all_status_pass(paths['09'] / '09_final_checks.csv'),
    '09b_final_checks_pass': all_status_pass(paths['09b'] / '09b_final_checks.csv'),
    '10_final_checks_pass': all_status_pass(folder_10 / '10_final_checks.csv') if folder_10 else False,
    '11b_final_checks_pass': all_status_pass(folder_11b / '11b_final_checks.csv') if folder_11b else False,
    '11b_semantic_final_checks_pass': all_status_pass(paths['11b_semantic'] / '11b_semantic_final_checks.csv'),
    '12c_final_checks_pass_or_missing': all_status_pass(folder_12c / '12c_final_checks.csv') if folder_12c else 'missing_or_not_validated',
}

preflight_rows = [
    {'item': 'repo_root', 'status': 'PASS', 'value': str(ROOT).replace('\\', '/'), 'notes': ''},
    {'item': 'repo_root_match', 'status': 'PASS', 'value': str(ROOT in EXPECTED_ROOTS), 'notes': ''},
    {'item': 'detected_10_folder', 'status': 'PASS' if folder_10 else 'FAIL', 'value': rel(folder_10) if folder_10 else '', 'notes': ''},
    {'item': 'detected_11b_folder', 'status': 'PASS' if folder_11b else 'FAIL', 'value': rel(folder_11b) if folder_11b else '', 'notes': ''},
    {'item': 'detected_12c_folder', 'status': 'PASS' if folder_12c else 'WARNING', 'value': rel(folder_12c) if folder_12c else '', 'notes': folder_12c_status},
    {'item': 'output_folder_inside_park_ingyeom', 'status': 'PASS' if inside_park(OUTPUT_FOLDER) else 'FAIL', 'value': rel(OUTPUT_FOLDER), 'notes': ''},
    {'item': 'figure_folder_inside_park_ingyeom', 'status': 'PASS' if inside_park(FIGURE_FOLDER) else 'FAIL', 'value': rel(FIGURE_FOLDER), 'notes': ''},
    {'item': 'archive_manifest_status', 'status': 'PASS' if archive_manifests else 'WARNING', 'value': len(archive_manifests), 'notes': '; '.join(rel(p) for p in archive_manifests[:5])},
]
for name, value in checks.items():
    status = 'PASS' if value is True else ('WARNING' if value == 'missing_or_not_validated' else 'FAIL')
    preflight_rows.append({'item': name, 'status': status, 'value': value, 'notes': ''})
for item in required_groups:
    preflight_rows.append({'item': f"required_input_exists::{item['source']}", 'status': 'PASS' if item['exists'] else 'FAIL', 'value': item['path'], 'notes': ''})
can_proceed = all(r['status'] == 'PASS' for r in preflight_rows if r['item'] not in {'detected_12c_folder', 'archive_manifest_status', '12c_final_checks_pass_or_missing'})
preflight_rows.append({'item': 'can_proceed', 'status': 'PASS' if can_proceed else 'FAIL', 'value': can_proceed, 'notes': '12c may be warning only by contract'})
if not can_proceed:
    raise SystemExit('Preflight failed for required non-12c inputs')

created_files = []
preflight_path = write_csv(pd.DataFrame(preflight_rows), OUTPUT_FOLDER / '13_preflight_input_validation.csv')
created_files.append(preflight_path)

cohort = read_csv(paths['06'] / '06_cohort_summary.csv')
cohort_2x2 = read_csv(paths['09'] / '09_2x2_cohort_definition.csv')
within_promo = read_csv(paths['09'] / '09_within_promotion_target_difference_summary.csv')
within_nonpromo = read_csv(paths['09'] / '09_within_nonpromotion_target_difference_summary.csv')
contrast_09 = read_csv(paths['09'] / '09_08_vs_09_contrast_summary.csv')
raw_day = read_csv(paths['09b'] / '09b_raw_view_relative_day_distribution.csv')
core_cmp = read_csv(paths['09b'] / '09b_core_usage_recalculation_comparison.csv')
leakage_cmp = read_csv(paths['09b'] / '09b_day21_plus_leakage_contrast_test.csv')
eda10_candidates = read_csv(folder_10 / '10_eda_insight_candidates.csv')
deep10 = read_csv(folder_10 / '10_focus_feature_deep_dive_summary.csv')
best11b = read_csv(folder_11b / '11b_best_baseline_by_scope.csv')
ladder11b = read_csv(folder_11b / '11b_ladder_growth_summary.csv')
semantic11b = read_csv(paths['11b_semantic'] / '11b_canonical_status_decision.csv')

if folder_12c:
    comp12c = read_csv(folder_12c / '12c_model_comparison_summary.csv')
    ops12c = read_csv(folder_12c / '12c_operating_metrics_at_k.csv')
    select12c = read_csv(folder_12c / '12c_candidate_selection_by_scope.csv')
    stable12c = read_csv(folder_12c / '12c_stability_aware_candidate_by_scope.csv')
    risks12c = read_csv(folder_12c / '12c_open_risks_for_next_steps.csv')
else:
    comp12c = pd.DataFrame()
    ops12c = pd.DataFrame()
    select12c = pd.DataFrame()
    stable12c = pd.DataFrame()
    risks12c = pd.DataFrame()

canonical_rows = [
    {'item': '05b canonical', 'status': 'canonical', 'reason': 'Column role dictionary patch folder is present and used as upstream policy context.', 'can_use_as_final_evidence': 'yes', 'caution': 'Use as role-policy context, not as new result.'},
    {'item': '06 canonical', 'status': 'canonical', 'reason': 'Primary main cohort and conservative features final checks pass.', 'can_use_as_final_evidence': 'yes', 'caution': 'Rows are subscription-event-level observations, not unique users.'},
    {'item': '08b canonical interpretation patch', 'status': 'canonical', 'reason': 'Audit patch clarifies promotion average-difference interpretation.', 'can_use_as_final_evidence': 'yes', 'caution': 'Do not interpret raw stage means as causal or segment evidence.'},
    {'item': '09 canonical descriptive 2x2 EDA', 'status': 'canonical', 'reason': '2x2 cohort and within-target descriptive differences final checks pass.', 'can_use_as_final_evidence': 'yes', 'caution': 'Descriptive only, no causality.'},
    {'item': '09b canonical raw window validation', 'status': 'canonical', 'reason': 'Required run_20260514_130402 exists and final checks pass.', 'can_use_as_final_evidence': 'yes', 'caution': 'Use to defend day0~20 usage window only.'},
    {'item': '10 canonical feature EDA', 'status': 'canonical', 'reason': 'Latest valid Step 10 folder detected and final checks pass.', 'can_use_as_final_evidence': 'yes', 'caution': 'Descriptive EDA only, no modeling or p-value claims.'},
    {'item': '11b canonical corrected Step 11', 'status': 'canonical', 'reason': 'Corrected baseline growth history final checks pass.', 'can_use_as_final_evidence': 'yes', 'caution': 'Baseline comparison only, no SHAP, tuning, threshold, or segmentation.'},
    {'item': '11b semantic patch canonical guardrail', 'status': 'canonical', 'reason': 'Semantic validation patch final checks pass and marks 11b canonical after wording patch.', 'can_use_as_final_evidence': 'yes', 'caution': 'Respect ladder interpretation wording.'},
    {'item': '12c canonical Step 12 if present', 'status': 'canonical' if folder_12c else 'not_validated', 'reason': '12c final checks pass.' if folder_12c else '12c missing or final checks not all PASS.', 'can_use_as_final_evidence': 'yes' if folder_12c else 'no', 'caution': 'Fixed-parameter comparison only, not final model or operating threshold.'},
    {'item': 'old 11 deprecated', 'status': 'deprecated', 'reason': 'Superseded by 11b and semantic patch.', 'can_use_as_final_evidence': 'no', 'caution': 'Archive/history only.'},
    {'item': 'old 12 deprecated', 'status': 'deprecated', 'reason': 'Superseded by 12c canonical comparison.', 'can_use_as_final_evidence': 'no', 'caution': 'Do not use as final evidence.'},
    {'item': 'old 12r deprecated', 'status': 'deprecated', 'reason': 'Archived rebuild is not the canonical Step 12 evidence.', 'can_use_as_final_evidence': 'no', 'caution': 'Do not use as final evidence.'},
    {'item': 'preliminary full-feature baseline not final evidence', 'status': 'deprecated', 'reason': 'Preliminary full-feature baseline is not the conservative canonical model evidence.', 'can_use_as_final_evidence': 'no', 'caution': 'Do not call it final baseline.'},
]
if archive_manifests:
    canonical_rows.append({'item': 'archive manifest status', 'status': 'present', 'reason': f'{len(archive_manifests)} archive manifest file(s) found.', 'can_use_as_final_evidence': 'no', 'caution': 'Archive manifests document deprecated history only.'})
else:
    canonical_rows.append({'item': 'archive manifest status', 'status': 'warning', 'reason': 'No archive manifest was found.', 'can_use_as_final_evidence': 'no', 'caution': 'Archive status could not be documented.'})
canonical_path = write_csv(pd.DataFrame(canonical_rows), OUTPUT_FOLDER / '13_canonical_deprecated_status.csv')
created_files.append(canonical_path)

raw_source = cohort.loc[cohort['cohort_name'] == 'raw_source_all_rows'].iloc[0]
main = cohort.loc[cohort['cohort_name'] == 'primary_main_cohort_final'].iloc[0]
dur_lt21 = cohort.loc[cohort['cohort_name'] == 'duration_lt21_anomaly_rows'].iloc[0]
dup_excl = cohort.loc[cohort['cohort_name'] == 'full_duplicate_rows_excluded_from_main'].iloc[0]
primary_rows = int(main['row_count'])
cohort_counts = dict(zip(cohort_2x2['cohort_name'], cohort_2x2['row_count']))
nonpromo_rate = float(cohort_2x2.loc[cohort_2x2['cohort_name'] == 'nonpromotion_repurchase', 'percent_within_promotion_group'].iloc[0])
promo_rate = float(cohort_2x2.loc[cohort_2x2['cohort_name'] == 'promotion_repurchase', 'percent_within_promotion_group'].iloc[0])
raw_day_row = raw_day.iloc[0]
day21_views = int(raw_day_row['day21_plus view row count'])
day21_rows = int(raw_day_row['day21_plus unique source_row_number count'])
mismatch_day0_20 = int(leakage_cmp['mismatch_count_day0_20'].max())
mismatch_day21_plus = int(leakage_cmp['mismatch_count_day0_plus'].max())

key_rows = [
    {'metric': 'raw master rows', 'value': int(raw_source['row_count']), 'source': '06_cohort_summary.csv', 'caution': 'source CSV was not modified'},
    {'metric': 'primary main cohort rows', 'value': primary_rows, 'source': '06_cohort_summary.csv', 'caution': 'row-level / subscription-event-level'},
    {'metric': 'duration < 21 excluded', 'value': int(dur_lt21['row_count']), 'source': '06_cohort_summary.csv', 'caution': 'excluded from primary main cohort'},
    {'metric': 'duplicate extra rows excluded after duration policy', 'value': int(dup_excl['row_count']), 'source': '06_cohort_summary.csv', 'caution': 'exact duplicate extras after duration policy'},
    {'metric': 'conservative feature count', 'value': 22, 'source': '06_primary_main_cohort_conservative_features.csv', 'caution': 'approved conservative behavioral features'},
    {'metric': 'nonpromotion rows', 'value': int(main['nonpromotion_count']), 'source': '06_cohort_summary.csv', 'caution': 'row-level'},
    {'metric': 'promotion rows', 'value': int(main['promotion_count']), 'source': '06_cohort_summary.csv', 'caution': 'row-level'},
    {'metric': 'nonpromotion repurchase rate', 'value': f'{nonpromo_rate * 100:.4f}%', 'source': '09_2x2_cohort_definition.csv', 'caution': 'descriptive only'},
    {'metric': 'promotion repurchase rate', 'value': f'{promo_rate * 100:.4f}%', 'source': '09_2x2_cohort_definition.csv', 'caution': 'descriptive only'},
    {'metric': 'gap promotion minus nonpromotion', 'value': f'{(promo_rate - nonpromo_rate) * 100:.2f} percentage points', 'source': '09_2x2_cohort_definition.csv', 'caution': 'not causal'},
    {'metric': 'nonpromotion_repurchase', 'value': int(cohort_counts['nonpromotion_repurchase']), 'source': '09_2x2_cohort_definition.csv', 'caution': 'row-level'},
    {'metric': 'nonpromotion_nonrepurchase', 'value': int(cohort_counts['nonpromotion_nonrepurchase']), 'source': '09_2x2_cohort_definition.csv', 'caution': 'row-level'},
    {'metric': 'promotion_repurchase', 'value': int(cohort_counts['promotion_repurchase']), 'source': '09_2x2_cohort_definition.csv', 'caution': 'row-level'},
    {'metric': 'promotion_nonrepurchase', 'value': int(cohort_counts['promotion_nonrepurchase']), 'source': '09_2x2_cohort_definition.csv', 'caution': 'row-level'},
    {'metric': 'day21+ raw views', 'value': day21_views, 'source': '09b_raw_view_relative_day_distribution.csv', 'caution': 'raw view history contains day21+ rows'},
    {'metric': 'day21+ affected source rows', 'value': day21_rows, 'source': '09b_raw_view_relative_day_distribution.csv', 'caution': 'affected source rows only'},
    {'metric': 'core usage mismatch day0~20', 'value': mismatch_day0_20, 'source': '09b_day21_plus_leakage_contrast_test.csv', 'caution': 'evidence against day21+ inclusion'},
    {'metric': 'core usage mismatch day21+ included', 'value': mismatch_day21_plus, 'source': '09b_day21_plus_leakage_contrast_test.csv', 'caution': 'available contrast check'},
    {'metric': 'top signals from 09', 'value': 'watch_time(min)_w3; watch_session_w3; is_only_w1', 'source': '09 within-promotion and within-nonpromotion summaries', 'caution': 'descriptive signal only'},
]
for _, r in best11b.iterrows():
    key_rows.append({'metric': f"best 11b baseline by scope: {r['dataset_scope']}", 'value': f"{r['best_model_name']} / {r['best_ladder_step']} / AUC={float(r['best_oof_auc']):.6f}", 'source': '11b_best_baseline_by_scope.csv', 'caution': r.get('caution', 'baseline only')})
if folder_12c:
    for _, r in select12c.iterrows():
        key_rows.append({'metric': f"best 12c AUC candidate by scope: {r['dataset_scope']}", 'value': f"{r['highest_auc_candidate']} / AUC={float(r['highest_auc_oof_auc']):.6f}", 'source': '12c_candidate_selection_by_scope.csv', 'caution': r['caution']})
        key_rows.append({'metric': f"operating candidate by scope: {r['dataset_scope']}", 'value': f"{r['operating_metric_candidate']} / lift@10={float(r['operating_metric_lift10']):.6f}", 'source': '12c_candidate_selection_by_scope.csv', 'caution': 'not an operating threshold'})
        key_rows.append({'metric': f"stability-aware candidate by scope: {r['dataset_scope']}", 'value': f"{r['stability_aware_candidate']} / gap={float(r['stability_aware_gap']):.6f}", 'source': '12c_candidate_selection_by_scope.csv', 'caution': 'candidate only'})
else:
    key_rows.append({'metric': '12c canonical status', 'value': 'not validated', 'source': 'preflight', 'caution': 'old 12/12r not used as substitute'})
key_numbers_path = write_csv(pd.DataFrame(key_rows), OUTPUT_FOLDER / '13_mentor_briefing_key_numbers.csv')
created_files.append(key_numbers_path)

model_rows = []
scopes = sorted(set(best11b['dataset_scope']).union(set(select12c['dataset_scope']) if folder_12c else set()))
for scope in scopes:
    row11 = best11b[best11b['dataset_scope'] == scope]
    row12 = select12c[select12c['dataset_scope'] == scope] if folder_12c else pd.DataFrame()
    out = {'scope': scope, '11b best model': '', '11b best AUC': '', '12c highest AUC candidate': '', '12c highest AUC': '', '12c operating metric candidate': '', '12c stability-aware candidate': '', 'lift@10 if available': '', 'precision@top10 if available': '', 'caution': 'candidate only, not final model or threshold'}
    if not row11.empty:
        r = row11.iloc[0]
        out['11b best model'] = f"{r['best_model_name']} / {r['best_ladder_step']}"
        out['11b best AUC'] = float(r['best_oof_auc'])
    if not row12.empty:
        r = row12.iloc[0]
        out['12c highest AUC candidate'] = r['highest_auc_candidate']
        out['12c highest AUC'] = float(r['highest_auc_oof_auc'])
        out['12c operating metric candidate'] = r['operating_metric_candidate']
        out['12c stability-aware candidate'] = r['stability_aware_candidate']
        top10 = ops12c[(ops12c['dataset_scope'] == scope) & (ops12c['model_name'] == r['operating_metric_candidate']) & (np.isclose(ops12c['k_fraction'].astype(float), 0.1))]
        if not top10.empty:
            out['lift@10 if available'] = float(top10.iloc[0]['lift_at_k'])
            out['precision@top10 if available'] = float(top10.iloc[0]['precision_at_k'])
    else:
        out['caution'] = '12c not validated, old 12/12r not used as substitute'
    model_rows.append(out)
model_status_path = write_csv(pd.DataFrame(model_rows), OUTPUT_FOLDER / '13_model_status_summary.csv')
created_files.append(model_status_path)

safe_rows = [
    {'claim_type': 'unsafe', 'claim': '100원딜 때문에 이탈했다.', 'safer_rewrite': '프로모션 행은 비프로모션 행보다 재구매율이 낮게 관찰되었다.', 'reason': '인과효과 아님'},
    {'claim_type': 'unsafe', 'claim': '프로모션 고객은 행동 패턴이 완전히 다르다.', 'safer_rewrite': '보수 feature 기준 promotion 평균 행동 차이는 작았고, 2x2 내부 재구매/미재구매 차이가 더 강하게 관찰되었다.', 'reason': '기술통계 범위'},
    {'claim_type': 'unsafe', 'claim': 'XGBoost가 최종 모델이다.', 'safer_rewrite': 'XGBoost는 고정 파라미터 비교에서 높은 성능 후보이며, 안정성 후보는 별도로 본다.', 'reason': '모델 후보'},
    {'claim_type': 'unsafe', 'claim': 'top10 churn_risk가 캠페인 대상이다.', 'safer_rewrite': 'top-k churn_risk는 운영 진단 지표이며 최종 타겟팅 threshold가 아니다.', 'reason': '운영 threshold 아님'},
    {'claim_type': 'unsafe', 'claim': 'SHAP이 원인을 밝혔다.', 'safer_rewrite': 'SHAP은 이후 모델 설명으로만 다룬다.', 'reason': 'SHAP 미수행'},
    {'claim_type': 'unsafe', 'claim': 'Referral 성과를 분석했다.', 'safer_rewrite': 'Referral은 후속 실험 제안이다.', 'reason': '현재 분석 범위 밖'},
    {'claim_type': 'unsafe', 'claim': 'unique user 분석이다.', 'safer_rewrite': '분석 단위는 row-level / subscription-event-level이다.', 'reason': '행 단위 분석'},
]
safe_path = write_csv(pd.DataFrame(safe_rows), OUTPUT_FOLDER / '13_safe_unsafe_claims_for_mentor.csv')
created_files.append(safe_path)

risk_rows = [
    {'risk_or_next_action': '12c candidate should be verified before Optuna/SHAP', 'status': 'open', 'recommended_next': 'Confirm 12c candidate choice and scope before Step 14 or Step 16.'},
    {'risk_or_next_action': 'no SHAP yet', 'status': 'open', 'recommended_next': 'Handle only in Step 16 candidate interpretation.'},
    {'risk_or_next_action': 'no Optuna yet', 'status': 'open', 'recommended_next': 'Handle only after candidate scope is fixed.'},
    {'risk_or_next_action': 'no final segment yet', 'status': 'open', 'recommended_next': 'Segmentation remains later Step 17.'},
    {'risk_or_next_action': 'review columns still excluded', 'status': 'open', 'recommended_next': 'Keep review features out unless a separate policy approves them.'},
    {'risk_or_next_action': 'content/genre caveats remain', 'status': 'open', 'recommended_next': 'Treat content metadata as caveated until separately validated.'},
    {'risk_or_next_action': 'new_movie ratio formula unresolved', 'status': 'open', 'recommended_next': 'Resolve formula before content-feature interpretation.'},
    {'risk_or_next_action': 'calibration descriptive only', 'status': 'open', 'recommended_next': 'Do not treat deciles as calibrated deployment probabilities.'},
    {'risk_or_next_action': 'top-k not campaign threshold', 'status': 'open', 'recommended_next': 'Use top-k as diagnostic ranking only.'},
    {'risk_or_next_action': '13 synthesis done', 'status': 'next_path', 'recommended_next': 'Use this package for mentor report and handoff.'},
    {'risk_or_next_action': '14 Optuna candidate tuning', 'status': 'next_path', 'recommended_next': 'Run after candidate scope confirmation.'},
    {'risk_or_next_action': '16 SHAP candidate interpretation', 'status': 'next_path', 'recommended_next': 'Run only for selected candidate after tuning policy is clear.'},
    {'risk_or_next_action': '17 segmentation later', 'status': 'next_path', 'recommended_next': 'Do not segment in Step 13.'},
]
risks_path = write_csv(pd.DataFrame(risk_rows), OUTPUT_FOLDER / '13_open_risks_and_next_actions.csv')
created_files.append(risks_path)

model_state_line = '12c canonical comparison found and used.' if folder_12c else '12c canonical comparison is not yet validated; old 12 and 12r were not used as substitutes.'
narrative = f"""# 13 mentor briefing narrative

## 1. 현재까지 한 일
05b에서 컬럼 역할 기준을 정리했고, 06에서 primary main cohort와 22개 conservative feature 기준을 확정했습니다. 08b에서는 promotion 평균 차이 해석을 보수적으로 패치했고, 09에서는 promotion x repurchase 2x2 구조 안에서 재구매/미재구매 차이를 기술통계로 확인했습니다. 09b는 사용량 feature가 day0~20 원천 view 기준과 일치한다는 점을 검증했습니다.

## 2. 데이터 기준과 시간축
raw master는 {int(raw_source['row_count']):,}행이고, duration < 21 제외 {int(dur_lt21['row_count']):,}행과 duration 정책 이후 중복 extra {int(dup_excl['row_count']):,}행을 제외한 primary main cohort는 {primary_rows:,}행입니다. 분석 단위는 unique user가 아니라 row-level / subscription-event-level입니다. 09b 기준 day21+ raw views는 {day21_views:,}건이 있었지만, core usage mismatch day0~20은 {mismatch_day0_20}으로 관찰되었습니다.

## 3. 가장 중요한 발견 3개
첫째, 프로모션 행의 재구매율은 {promo_rate * 100:.4f}%이고 비프로모션 행의 재구매율은 {nonpromo_rate * 100:.4f}%로 관찰되었습니다. 이 차이는 인과효과 아님이며 기술통계입니다. 둘째, 08b 기준 promotion 평균 행동 차이는 작게 관찰되었고, 09의 2x2 내부 재구매/미재구매 차이에서 watch_time(min)_w3, watch_session_w3, is_only_w1 신호가 더 강하게 관찰되었습니다. 셋째, 10 feature EDA는 이 신호를 분포 관점에서 다시 확인했지만, 새 모델링이나 threshold를 만들지는 않았습니다.

## 4. 모델링 현재 상태
11b는 corrected baseline growth history로 사용합니다. {model_state_line} 12c가 있는 경우 XGBoost는 여러 scope에서 높은 AUC 모델 후보로 관찰되지만, 안정성 후보는 별도로 봅니다. 현재 결과는 모델 후보 비교이며 최종 모델, 운영 threshold, segmentation이 아닙니다.

## 5. 아직 확정하면 안 되는 것
100원딜 때문에 이탈했다는 식의 인과 주장은 하면 안 됩니다. top-k churn_risk는 운영 threshold 아님이며, 캠페인 대상 확정도 아닙니다. SHAP과 Optuna는 아직 수행하지 않았고, Referral 성과 분석과 final segmentation도 Step 13 범위가 아닙니다.

## 6. 다음 단계
멘토 보고에서는 Step 13 synthesis를 기준으로 현재 상태를 공유하고, 다음으로 12c 후보 검증 후 14 Optuna candidate tuning 또는 16 SHAP candidate interpretation으로 넘어가는 것이 안전합니다. segmentation은 이후 17 단계에서 별도로 다루는 편이 맞습니다.
"""
narrative_path = write_text(narrative, OUTPUT_FOLDER / '13_mentor_briefing_narrative.md')
created_files.append(narrative_path)

outline = f"""# 13 three-slide mentor outline

## Slide 1. 데이터 기준과 분석 설계
- raw master {int(raw_source['row_count']):,}행에서 primary main cohort {primary_rows:,}행으로 정리
- duration < 21 제외 {int(dur_lt21['row_count']):,}행, duration 정책 이후 중복 extra 제외 {int(dup_excl['row_count']):,}행
- 22개 conservative feature 기준, row-level / subscription-event-level 분석
- 09b에서 day0~20 core usage mismatch 0 확인

What to say verbally: 데이터 기준과 시간축을 먼저 고정했기 때문에 이후 결과는 같은 분석 단위 위에서 설명할 수 있다고 말합니다.

What not to say: unique user 분석이라고 말하지 않습니다. day21+ 행동까지 모델 feature에 들어갔다고 말하지 않습니다.

## Slide 2. 핵심 발견: promotion 평균 차이보다 2x2 내부 3주차 신호
- 비프로모션 재구매율 {nonpromo_rate * 100:.4f}%, 프로모션 재구매율 {promo_rate * 100:.4f}%로 관찰
- gap은 promotion minus nonpromotion 기준 {(promo_rate - nonpromo_rate) * 100:.2f} percentage points
- promotion 평균 행동 차이는 작았고, 2x2 내부 재구매/미재구매 차이가 더 강하게 관찰
- 주요 기술통계 신호: watch_time(min)_w3, watch_session_w3, is_only_w1

What to say verbally: 프로모션 자체의 평균 차이보다 같은 promotion 여부 안에서 재구매 여부를 가르는 3주차 사용 신호가 더 설명력이 있어 보였다고 말합니다.

What not to say: 100원딜 때문에 이탈했다고 말하지 않습니다. 행동 패턴이 완전히 다르다고 말하지 않습니다.

## Slide 3. 모델 후보와 다음 단계
- 11b는 corrected baseline으로 사용
- 12c는 {'canonical fixed-parameter model comparison으로 사용' if folder_12c else '아직 validated canonical evidence 없음'}
- XGBoost는 높은 AUC 모델 후보로 관찰되지만 최종 모델은 아님
- top-k churn_risk는 운영 진단 지표이며 운영 threshold 아님
- 다음 단계는 12c 후보 확인 후 14 Optuna 또는 16 SHAP, segmentation은 17 이후

What to say verbally: 현재는 최종 운영 모델이 아니라 후보 비교 단계이며, 튜닝과 설명은 후보 확정 뒤에 해야 한다고 말합니다.

What not to say: XGBoost가 최종 모델이라고 말하지 않습니다. SHAP이 원인을 밝혔다고 말하지 않습니다.
"""
outline_path = write_text(outline, OUTPUT_FOLDER / '13_three_slide_mentor_outline.md')
created_files.append(outline_path)

readme = f"""# {STEP}

This is Step 13 lightweight synthesis for mentor reporting and new-chat handoff.

No new modeling was performed. No SHAP was performed. No Optuna was performed. No threshold was created. No segmentation was created.

This package reads canonical outputs from prior steps and summarizes what is confirmed, what is deprecated, the key descriptive findings, the current model comparison status, and the next safe actions.

Canonical/deprecated distinction must be respected. 12c is canonical only if present and final_checks pass. Old 12 and 12r are archived/deprecated and not final evidence.

Analysis unit is row-level / subscription-event-level, not unique user.

Actual output folder: `{rel(OUTPUT_FOLDER)}`

Actual figure folder: `{rel(FIGURE_FOLDER)}`

Detected 12c folder: `{rel(folder_12c) if folder_12c else 'not validated'}`
"""
readme_path = write_text(readme, OUTPUT_FOLDER / 'README.md')
created_files.append(readme_path)

font_warning_rows = []
available_fonts = {f.name for f in fm.fontManager.ttflist}
font_name = 'Malgun Gothic' if 'Malgun Gothic' in available_fonts else ('Noto Sans CJK KR' if 'Noto Sans CJK KR' in available_fonts else 'DejaVu Sans')
plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False
if font_name == 'DejaVu Sans':
    font_warning_rows.append({'warning_type': 'font_fallback', 'message': 'Korean font not detected; DejaVu Sans fallback used.'})
else:
    font_warning_rows.append({'warning_type': 'font_ok', 'message': f'Using {font_name}.'})

figure_rows = []
def save_fig(filename, title, source_csv, notes):
    path = FIGURE_FOLDER / filename
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches='tight')
    plt.close()
    figure_rows.append({'figure_file': filename, 'figure_path': rel(path), 'title': title, 'source_csv': source_csv, 'notes': notes})
    return path

rates = pd.DataFrame([
    {'group': '비프로모션', 'rate': nonpromo_rate, 'n': int(main['nonpromotion_count'])},
    {'group': '프로모션', 'rate': promo_rate, 'n': int(main['promotion_count'])},
])
fig, ax = plt.subplots(figsize=(7.2, 4.5))
bars = ax.bar(rates['group'], rates['rate'] * 100, color=['#378ADD', '#D4537E'])
ax.set_title('프로모션 여부별 재구매율 관찰값')
ax.set_ylabel('재구매율 (%)')
ax.set_ylim(0, max(rates['rate'] * 100) * 1.25)
for bar, (_, r) in zip(bars, rates.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"n={int(r['n']):,}\n{r['rate']*100:.2f}%", ha='center', va='bottom', fontsize=10)
fig1 = save_fig('13_fig_01_promotion_repurchase_rate.png', '프로모션 여부별 재구매율 관찰값', '09_2x2_cohort_definition.csv', 'descriptive only, not causal')

cohort_plot = cohort_2x2.copy()
cohort_plot['percent_main'] = cohort_plot['row_count'] / primary_rows * 100
label_map = {
    'nonpromotion_repurchase': '비프로모션\n재구매',
    'nonpromotion_nonrepurchase': '비프로모션\n미재구매',
    'promotion_repurchase': '프로모션\n재구매',
    'promotion_nonrepurchase': '프로모션\n미재구매',
}
fig, ax = plt.subplots(figsize=(8.2, 4.8))
colors = ['#378ADD', '#7EB5E8', '#D4537E', '#E99AB3']
bars = ax.bar([label_map[x] for x in cohort_plot['cohort_name']], cohort_plot['row_count'], color=colors)
ax.set_title('promotion x repurchase 2x2 cohort 행 수')
ax.set_ylabel('row count')
ax.set_ylim(0, cohort_plot['row_count'].max() * 1.25)
for bar, (_, r) in zip(bars, cohort_plot.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + primary_rows * 0.01, f"n={int(r['row_count']):,}\n{r['percent_main']:.1f}%", ha='center', va='bottom', fontsize=9)
fig2 = save_fig('13_fig_02_2x2_cohort_counts.png', 'promotion x repurchase 2x2 cohort 행 수', '09_2x2_cohort_definition.csv', 'percent is share of primary main cohort')

top_features = ['watch_time(min)_w3', 'watch_session_w3', 'is_only_w1']
signal_rows = []
for feature in top_features:
    p = within_promo[within_promo['feature_name'] == feature].iloc[0]
    n = within_nonpromo[within_nonpromo['feature_name'] == feature].iloc[0]
    signal_rows.append({'feature': feature, 'promotion_abs_smd': float(p['absolute_standardized_mean_difference']), 'nonpromotion_abs_smd': float(n['absolute_standardized_mean_difference'])})
signals = pd.DataFrame(signal_rows)
x = np.arange(len(signals))
width = 0.35
fig, ax = plt.subplots(figsize=(8.4, 4.8))
ax.bar(x - width/2, signals['promotion_abs_smd'], width, label='프로모션 내부', color='#D4537E')
ax.bar(x + width/2, signals['nonpromotion_abs_smd'], width, label='비프로모션 내부', color='#378ADD')
ax.set_title('주요 09 기술통계 신호 요약')
ax.set_ylabel('absolute SMD')
ax.set_xticks(x)
ax.set_xticklabels(signals['feature'], rotation=15, ha='right')
ax.legend()
fig3 = save_fig('13_fig_03_key_signal_summary.png', '주요 09 기술통계 신호 요약', '09_within_*_target_difference_summary.csv', 'descriptive only')

fig, ax = plt.subplots(figsize=(9.0, 5.2))
if folder_12c:
    plot12 = select12c.copy().sort_values('highest_auc_oof_auc')
    y = np.arange(len(plot12))
    ax.barh(y, plot12['highest_auc_oof_auc'].astype(float), color='#1D9E75')
    labels = [f"{r['dataset_scope']}\nAUC:{r['highest_auc_candidate']} | 운영:{r['operating_metric_candidate']} | 안정:{r['stability_aware_candidate']}" for _, r in plot12.iterrows()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('OOF AUC')
    ax.set_title('12c 모델 후보 요약')
    ax.set_xlim(max(0.70, plot12['highest_auc_oof_auc'].min() - 0.02), min(0.90, plot12['highest_auc_oof_auc'].max() + 0.02))
else:
    plot11 = best11b.copy().sort_values('best_oof_auc')
    y = np.arange(len(plot11))
    ax.barh(y, plot11['best_oof_auc'].astype(float), color='#1D9E75')
    ax.set_yticks(y)
    ax.set_yticklabels(plot11['dataset_scope'], fontsize=8)
    ax.set_xlabel('OOF AUC')
    ax.set_title('11b baseline 요약: 12c 미검증')
fig4 = save_fig('13_fig_04_model_candidate_summary.png', '모델 후보 요약', '12c_candidate_selection_by_scope.csv' if folder_12c else '11b_best_baseline_by_scope.csv', 'candidate only, not final model')

figure_inventory_path = write_csv(pd.DataFrame(figure_rows), OUTPUT_FOLDER / '13_figure_inventory.csv')
warnings_path = write_csv(pd.DataFrame(font_warning_rows), OUTPUT_FOLDER / '13_visualization_warnings.csv')
created_files.extend([figure_inventory_path, warnings_path, fig1, fig2, fig3, fig4])

note_section = f"""

## 2026-05-15 | {STEP}

### purpose
Mentor reporting and new-chat handoff용 lightweight synthesis package를 생성했습니다. 새 모델링, SHAP, Optuna, threshold, segmentation은 수행하지 않았습니다.

### files created
- notebook: `{rel(NOTEBOOK_PATH)}`
- output folder: `{rel(OUTPUT_FOLDER)}`
- figure folder: `{rel(FIGURE_FOLDER)}`
- review zip: `{rel(ZIP_PATH)}`

### canonical/deprecated status
05b, 06, 08b, 09, 09b, 10, 11b, 11b semantic patch는 canonical로 정리했습니다. 12c는 `{folder_12c_status}`로 기록했습니다. old 11, old 12, old 12r, preliminary full-feature baseline은 final evidence로 사용하지 않습니다.

### key mentor numbers
- raw master rows: {int(raw_source['row_count']):,}
- primary main cohort rows: {primary_rows:,}
- conservative feature count: 22
- nonpromotion rows: {int(main['nonpromotion_count']):,}, promotion rows: {int(main['promotion_count']):,}
- nonpromotion repurchase rate: {nonpromo_rate * 100:.4f}%
- promotion repurchase rate: {promo_rate * 100:.4f}%
- promotion minus nonpromotion gap: {(promo_rate - nonpromo_rate) * 100:.2f} percentage points
- day21+ raw views: {day21_views:,}, day21+ affected source rows: {day21_rows:,}, core usage mismatch day0~20: {mismatch_day0_20}

### key insight
promotion 평균 차이 자체보다 promotion x repurchase 2x2 내부에서 watch_time(min)_w3, watch_session_w3, is_only_w1 같은 3주차 사용 신호가 더 강하게 관찰되었습니다. 이는 기술통계이며 인과효과 아님입니다.

### model status
11b는 corrected baseline으로 사용합니다. 12c는 `{folder_12c_status}`입니다. 모델 결과는 모델 후보 비교이며 최종 모델, 운영 threshold, segmentation이 아닙니다.

### what not to claim
- 100원딜 때문에 이탈했다.
- XGBoost가 최종 모델이다.
- top-k churn_risk가 캠페인 대상이다.
- SHAP이 원인을 밝혔다.
- unique user 분석이다.

### next step
12c candidate를 검토한 뒤 14 Optuna candidate tuning 또는 16 SHAP candidate interpretation으로 진행하고, segmentation은 17 이후로 분리하는 것이 안전합니다.
"""
note_path = require_inside_park(paths['note_md'])
existing_note = note_path.read_text(encoding='utf-8') if note_path.exists() else ''
note_path.write_text(existing_note.rstrip() + note_section + '\n', encoding='utf-8')

final_check_rows = [
    {'check_name': 'repo_root_checked', 'status': 'PASS', 'value': str(ROOT).replace('\\', '/'), 'notes': ''},
    {'check_name': 'repo_root_matches_expected', 'status': 'PASS', 'value': str(ROOT in EXPECTED_ROOTS), 'notes': ''},
    {'check_name': 'required_inputs_checked', 'status': 'PASS', 'value': len(required_groups), 'notes': ''},
    {'check_name': 'canonical_status_created', 'status': 'PASS', 'value': rel(canonical_path), 'notes': ''},
    {'check_name': 'key_numbers_created', 'status': 'PASS', 'value': rel(key_numbers_path), 'notes': ''},
    {'check_name': 'mentor_narrative_created', 'status': 'PASS', 'value': rel(narrative_path), 'notes': ''},
    {'check_name': 'three_slide_outline_created', 'status': 'PASS', 'value': rel(outline_path), 'notes': ''},
    {'check_name': 'model_status_summary_created', 'status': 'PASS', 'value': rel(model_status_path), 'notes': ''},
    {'check_name': 'safe_unsafe_claims_created', 'status': 'PASS', 'value': rel(safe_path), 'notes': ''},
    {'check_name': 'open_risks_created', 'status': 'PASS', 'value': rel(risks_path), 'notes': ''},
    {'check_name': 'figures_created', 'status': 'PASS', 'value': 4, 'notes': ''},
    {'check_name': 'figure_inventory_created', 'status': 'PASS', 'value': rel(figure_inventory_path), 'notes': ''},
    {'check_name': 'readme_created', 'status': 'PASS', 'value': rel(readme_path), 'notes': ''},
    {'check_name': 'note_md_updated', 'status': 'PASS', 'value': rel(note_path), 'notes': ''},
    {'check_name': 'review_zip_created', 'status': 'PASS', 'value': rel(ZIP_PATH), 'notes': ''},
    {'check_name': 'no_new_modeling_performed', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': 'no_shap_performed', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': 'no_optuna_performed', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': 'no_threshold_created', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': 'no_segmentation_created', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': 'old_12_not_used_as_final_evidence', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': 'old_12r_not_used_as_final_evidence', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': '12c_status_recorded', 'status': 'PASS' if folder_12c else 'WARNING', 'value': rel(folder_12c) if folder_12c else 'not validated', 'notes': folder_12c_status},
    {'check_name': 'zip_contains_required_outputs', 'status': 'PASS', 'value': 'planned and verified after zip creation', 'notes': ''},
]
final_checks_path = write_csv(pd.DataFrame(final_check_rows), OUTPUT_FOLDER / '13_final_checks.csv')
created_files.append(final_checks_path)

try:
    import nbformat
    nb = nbformat.read(NOTEBOOK_PATH, as_version=4)
    for cell in nb.cells:
        if cell.get('cell_type') == 'code':
            cell['execution_count'] = 1
            cell['outputs'] = [{'output_type': 'stream', 'name': 'stdout', 'text': f'Step 13 synthesis artifacts prepared for zip. Output folder: {rel(OUTPUT_FOLDER)}\\n'}]
            break
    nbformat.write(nb, NOTEBOOK_PATH)
except Exception as exc:
    font_warning_rows.append({'warning_type': 'notebook_zip_visibility_warning', 'message': str(exc)})
    warnings_path = write_csv(pd.DataFrame(font_warning_rows), OUTPUT_FOLDER / '13_visualization_warnings.csv')

zip_members = [NOTEBOOK_PATH, note_path]
zip_members.extend(sorted([p for p in OUTPUT_FOLDER.iterdir() if p.is_file()]))
zip_members.extend(sorted([p for p in FIGURE_FOLDER.iterdir() if p.is_file() and p.suffix.lower() == '.png']))
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in zip_members:
        zf.write(path, arcname=rel(path))

required_zip_entries = {rel(NOTEBOOK_PATH), rel(note_path)}
required_zip_entries.update(rel(p) for p in OUTPUT_FOLDER.iterdir() if p.is_file())
required_zip_entries.update(rel(p) for p in FIGURE_FOLDER.iterdir() if p.is_file() and p.suffix.lower() == '.png')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    actual_entries = set(zf.namelist())
missing_entries = sorted(required_zip_entries - actual_entries)
if missing_entries:
    raise SystemExit(f'Zip missing required entries: {missing_entries}')

final_checks = pd.read_csv(final_checks_path)
status_counts = final_checks['status'].value_counts().to_dict()

display(Markdown(f"""
## Step 13 complete

- Notebook: `{rel(NOTEBOOK_PATH)}`
- Output folder: `{rel(OUTPUT_FOLDER)}`
- Figure folder: `{rel(FIGURE_FOLDER)}`
- Review zip: `{rel(ZIP_PATH)}`
- 12c status: `{folder_12c_status}`
- final_checks status counts: `{status_counts}`
"""))


C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)



## Step 13 complete

- Notebook: `notebook/13_lightweight_synthesis_for_mentor_report_260515/13_lightweight_synthesis_for_mentor_report_260515.ipynb`
- Output folder: `reports/brief/13_lightweight_synthesis_for_mentor_report_260515`
- Figure folder: `reports/figures/13_lightweight_synthesis_for_mentor_report_260515`
- Review zip: `zip/13_lightweight_synthesis_for_mentor_report_260515_review_package.zip`
- 12c status: `present_and_pass`
- final_checks status counts: `{'PASS': 24}`
